# Exploration of Databricks

### 1. Import Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, hour, avg, count

### 2. Ingest Data (Bronze Layer)

In [0]:
bronze_path = "/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2025-01.parquet"

# Read raw parquet into Spark DataFrame
df_bronze = (spark.read.parquet(bronze_path))
display(df_bronze.limit(5))  # Show sample rows
print("Bronze count:", df_bronze.count())


### 3. Clean & Transform Data (Silver Layer)

In [0]:
df_silver = (df_bronze
             .withColumn("pickup_datetime", col("tpep_pickup_datetime").cast("timestamp"))
             .withColumn("dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))
             .withColumn("passenger_count", col("passenger_count").cast("integer"))
             .withColumn("trip_distance", col("trip_distance").cast("double"))
             .withColumn("fare_amount", col("fare_amount").cast("double"))
             .filter(col("passenger_count") > 0)
             .filter(col("fare_amount") > 0))

display(df_silver.limit(5))
print("Silver count:", df_silver.count())

### 4. Curated Aggregations (Gold Layer)

In [0]:
df_gold = (df_silver
           .withColumn("pickup_hour", hour(col("pickup_datetime")))
           .groupBy("pickup_hour")
           .agg(avg("fare_amount").alias("avg_fare"),
                count("*").alias("trip_count"))
           .orderBy("pickup_hour"))

display(df_gold)

### 5. Pandas API on Spark Demo

In [0]:
import pyspark.pandas as ps

# Convert Spark DataFrame to pandas-on-Spark
psdf = df_silver.pandas_api()

# Pandas-style operations (executed on Spark cluster)
summary = psdf[["fare_amount", "trip_distance"]].describe()
display(summary)

# GroupBy example
avg_fares = psdf.groupby("passenger_count")["fare_amount"].mean()
display(avg_fares)

### 6. Persist Data in Delta Lake

In [0]:
# Save as a Delta table in the default catalog/schema
df_silver.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi_silver")

df_gold.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi_gold")

#### Demonstrating Spark SQL

In [0]:
%sql
SELECT * FROM nyc_taxi_silver LIMIT 10;

In [0]:
%sql

SELECT * FROM nyc_taxi_gold LIMIT 10;

### 7. Export Data to External Storage

In [0]:
export_path = "/Volumes/workspace/default/nyc_taxi/nyc_taxi_gold_export"
df_gold.write.mode("overwrite").parquet(export_path)